In [1]:
import numpy as np

def mn_rotation_to_quaternion(alpha_deg, beta_deg, gamma_deg, omega_m=0, omega_n=0):
    """
    Convert rotations and angular velocities in your m-n coordinate system
    
    alpha_deg: angle of n-axis relative to x-axis
    beta_deg: rotation about m-axis
    gamma_deg: rotation about n-axis
    omega_m: angular velocity about m-axis (rad/s)
    omega_n: angular velocity about n-axis (rad/s)
    
    Returns: (quaternion, angular_velocity_world, yz_plane_angle, yz_projection_factor, angle_to_ground_rad)
    """
    
    # Convert to radians
    alpha = np.radians(alpha_deg)
    beta = np.radians(beta_deg)
    gamma = np.radians(gamma_deg)
    
    # Define your m and n axes in terms of x,y,z coordinates
    # n-axis direction vector (in xy plane)
    n_axis = np.array([np.cos(alpha), np.sin(alpha), 0])
    
    # m-axis direction vector (perpendicular to n, in xy plane)
    # This is n-axis rotated by -90 degrees in xy plane
    m_axis = np.array([-np.sin(alpha), np.cos(alpha), 0])
    
    def axis_angle_to_quat(axis, angle):
        """Convert axis-angle to quaternion [w, x, y, z]"""
        axis = axis / np.linalg.norm(axis)  # normalize
        half_angle = angle / 2
        w = np.cos(half_angle)
        xyz = axis * np.sin(half_angle)
        return np.array([w, xyz[0], xyz[1], xyz[2]])
    
    def quat_multiply(q1, q2):
        """Multiply two quaternions q1 * q2"""
        w1, x1, y1, z1 = q1
        w2, x2, y2, z2 = q2
        return np.array([
            w1*w2 - x1*x2 - y1*y2 - z1*z2,
            w1*x2 + x1*w2 + y1*z2 - z1*y2,
            w1*y2 - x1*z2 + y1*w2 + z1*x2,
            w1*z2 + x1*y2 - y1*x2 + z1*w2
        ])
    
    # Create quaternions for each rotation
    # First rotate about m-axis by beta
    q_m = axis_angle_to_quat(m_axis, beta)
    
    # Then rotate about n-axis by gamma
    q_n = axis_angle_to_quat(n_axis, gamma)
    
    # Combine rotations: first m, then n
    # Note: quaternion multiplication is in reverse order (right to left)
    final_quat = quat_multiply(q_n, q_m)
    
    # To find the rod orientation, we need to rotate the local z-axis
    # Using rotation matrix might be clearer for analysis
    def quat_to_rotation_matrix(q):
        """Convert quaternion to rotation matrix"""
        w, x, y, z = q
        return np.array([
            [1-2*(y*y+z*z), 2*(x*y-w*z), 2*(x*z+w*y)],
            [2*(x*y+w*z), 1-2*(x*x+z*z), 2*(y*z-w*x)],
            [2*(x*z-w*y), 2*(y*z+w*x), 1-2*(x*x+y*y)]
        ])
    
    # Get the rotation matrix
    R = quat_to_rotation_matrix(final_quat)
    
    # Rod's local axis (along z in local coordinates)
    local_rod_axis = np.array([0, 0, 1])
    
    # Transform to world coordinates
    world_rod_axis = R @ local_rod_axis
    
    # Calculate angle in yz plane
    # This is the angle from the positive z-axis to the projection in yz plane
    yz_angle_rad = np.arctan2(world_rod_axis[1], world_rod_axis[2])
    yz_angle_deg = np.degrees(yz_angle_rad)
    
    # Calculate projection factor in yz plane
    yz_projection_factor = np.sqrt(world_rod_axis[1]**2 + world_rod_axis[2]**2)
    
    # Angle to ground (from horizontal plane)
    # This is the angle between the rod and the xy plane
    angle_to_ground_rad = np.arcsin(np.clip(world_rod_axis[2], -1, 1))
    
    # Calculate angular velocity in world frame
    omega_world = omega_m * m_axis + omega_n * n_axis
    
    return final_quat, omega_world, yz_angle_deg, yz_projection_factor, angle_to_ground_rad

In [2]:
import mujoco
import mujoco.viewer
import time
import numpy as np

def create_world_with_wall_height(wall_height):
    import re
    with open("yepers.xml", 'r') as f:
        xml_content = f.read()
    
    wall_half_height = wall_height / 2
    wall_center_z = wall_half_height

    # Calculate side_detector height (wall_height - 0.1)
    wall_detector_height = wall_height - 0.1
    wall_detector_half_height = wall_detector_height / 2
    wall_detector_center_z = wall_detector_half_height

    # Update wall - match the entire line
    wall_pattern = r'<geom name="wall"[^>]*>'
    new_wall = f'<geom name="wall" type="box" size="5 0.03 {wall_half_height}" pos="0 0 {wall_center_z}" rgba="0.8 0.4 0.2 1"/>'
    modified_xml = re.sub(wall_pattern, new_wall, xml_content)

    # Update side_detector - match the entire line
    wall_detector_pattern = r'<geom name="side_detector"[^>]*>'
    new_wall_detector = f'<geom name="side_detector" type="box" size="5 0.02 {wall_detector_half_height}" pos="0 -0.05 {wall_detector_center_z}" rgba="0 1 0 0.3" contype="1" conaffinity="1"/>'
    modified_xml = re.sub(wall_detector_pattern, new_wall_detector, modified_xml)

    # Update far_side_detector - match the entire line
    far_detector_pattern = r'<geom name="far_side_detector"[^>]*>'
    new_far_detector = f'<geom name="far_side_detector" type="box" size="5 2 0.1" pos="0 -2 0.1" rgba="0 1 0 0.3" contype="1" conaffinity="1"/>'
    modified_xml = re.sub(far_detector_pattern, new_far_detector, modified_xml)

    model = mujoco.MjModel.from_xml_string(modified_xml)
    return model

def check_wall_contact(model, data):
    
    rod_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "rod")
    
    # Check all contacts in the simulation
    for i in range(data.ncon):
        contact = data.contact[i]
        
        # Get the geom IDs involved in this contact
        geom1_id = contact.geom1
        geom2_id = contact.geom2
        
        # Get the body IDs for these geometries
        body1_id = model.geom_bodyid[geom1_id]
        body2_id = model.geom_bodyid[geom2_id]
        
        # Check if one of the bodies is the rod
        if body1_id == rod_body_id or body2_id == rod_body_id:
            # Get the names of the geometries to identify what the rod is touching
            geom1_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_GEOM, geom1_id)
            geom2_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_GEOM, geom2_id)
            
            # Check if either geometry is a wall (assuming walls have "wall" in their name)
            if (geom1_name and "wall" in geom1_name.lower()) or \
               (geom2_name and "wall" in geom2_name.lower()):
                return 0  # Contact with wall detected
    return 1

def run_rod_simulation(hv, vv, d, a, b, avm, g, avn, wall_height):
    
    # Load the world model
    model = create_world_with_wall_height(wall_height)
    data = mujoco.MjData(model)

    rod_geom_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, "rod_geom")
    rod_half_length = model.geom_size[rod_geom_id][1]

    quaternion, omega, yz_angle, yz_projection_factor, angle_to_ground_rad = mn_rotation_to_quaternion(a, b, g, avm, avn)
    half_projection_length = rod_half_length * yz_projection_factor

    data.qpos[0:3] = [-1, d + half_projection_length * np.cos(np.deg2rad(90 - yz_angle)), rod_half_length*np.sin(angle_to_ground_rad) + 0.03]
    data.qpos[3:7] = quaternion
    data.qvel[0:3] = [hv*np.cos(np.deg2rad(a)), hv*np.sin(np.deg2rad(a)), vv]
    data.qvel[3:6] = omega

    # Create viewer with keyboard callback
    paused = True

    def key_callback(keycode):
        nonlocal paused  # Use nonlocal instead of global
        if keycode == 32:  # Spacebar key code
            paused = not paused
            if paused:
                print("Simulation PAUSED - Press SPACEBAR to resume")
            else:
                print("Simulation STARTED - Press SPACEBAR to pause")

    # Launch the viewer (non-passive for keyboard support)
    with mujoco.viewer.launch_passive(model, data, key_callback=key_callback) as viewer:
        # Set camera to look down and forward at the bar
        viewer.cam.lookat[0] = 0    # Look at X=0 (center of bar)
        viewer.cam.lookat[1] = 0    # Look at Y=0 (center of bar)  
        viewer.cam.lookat[2] = 1.5  # Look at Z=1.5 (slightly below bar)
        
        viewer.cam.distance = 18     # Distance from the look-at point
        viewer.cam.elevation = -20  # Look down at -20 degrees
        viewer.cam.azimuth = 0      # Face forward (0 degrees)
        

        print("Rod is ready! Press SPACEBAR in the viewer window to start/pause simulation")
        
        while viewer.is_running():
            if not paused:
                mujoco.mj_step(model, data)
                wall_contact_status = check_wall_contact(model, data)
                if wall_contact_status == 0:
                    print("Rod has contacted a wall! Simulation paused.")
                    paused = True
            viewer.sync()
            time.sleep(0.01)  # Prevent excessive CPU usage when paused

In [3]:
# Usage:
run_rod_simulation(
    hv=7,
    vv=-4.3,
    d=1,
    a=-45,
    b=-20,
    avm=5,
    g=-20,
    avn=5,
    wall_height = 1.8
)

ValueError: could not broadcast input array from shape (4,) into shape (3,)